# 실습 3. OpenAI API 활용 프로그램 (Day 2 / 120분)

> 시나리오: **리뷰 감성 분류를 LLM 으로 다시 풀고, sklearn(실습 2)과 비교**
>
> 이 노트북은 외부 예제 없이 `openai` 패키지만으로 진행합니다.

## Task
1. 단발 호출 / 토큰·비용 / 스트리밍 (`.env` 의 `OPENAI_API_KEY`)
2. 리뷰 100개 긍/부정 분류 — `temperature=0`, JSON 응답 강제 → **실습 2 와 비교**
3. 비용 측정 (`response.usage`) → **1만 건 가정 시 비용 추산**

## 0) 준비 — `.env` 로드 + 클라이언트

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()                 # .env 의 OPENAI_API_KEY 를 환경변수로 로드
client = OpenAI()             # 키는 환경변수에서 자동으로 읽음
MODEL = "gpt-4o-mini"
print("client ready:", MODEL)

## 2) Task 2 — 리뷰 100개 긍/부정 분류 (LLM)

실습 1의 산출물 `reviews_clean.parquet` 에서 100건을 샘플링해 분류한다.
- `temperature=0` (분류는 결정적이어야 함)
- **JSON 응답 강제** (`response_format`) — 파싱 안전

### sklearn(실습 2) 과 비교 — 정확도

## 3) Task 3 — 비용 측정 + 1만 건 추산

## 회고 / 산출물
- [ ] 비교표: ML vs LLM (정확도 / 비용 / 속도 / 일관성)
- [ ] 한 줄 결론 — *대량·단순 분류는 ___, 유연·복잡 추론은 ___*

In [ ]:
반어_리뷰 = [
    '와 정말 좋네요 한 번 쓰고 바로 고장나서 아주 만족합니다',
    '품질 최고예요~ 환불하고 싶을 만큼',
    '빠른 배송 감사합니다 일주일이나 걸려서요',
]

import json

SYSTEM2 = (
    '너는 한국어 쇼핑몰 리뷰 감성 분류기다. 입력 리뷰가 긍정이면 1, 부정이면 0.',
    '또한 리뷰의 "화" 수치를 0에서 100 사이의 정수로 반환한다.',
    '오직 JSON으로만 답하라: {\"label\": 0 또는 1, \"anger\": 정수}.',
    '추가 설명이나 텍스트를 덧붙이지 마라.',
)

def classify_with_anger(text: str):
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": SYSTEM2},
            {"role": "user", "content": text},
        ],
    )
    data = json.loads(resp.choices[0].message.content)
    label = int(data.get("label", 0))
    anger = int(data.get("anger", 0))
    return label, anger, resp.usage.total_tokens

results = []
tokens = 0
for t in 반어_리뷰:
    try:
        label, anger, used = classify_with_anger(t)
    except Exception as e:
        label, anger, used = None, None, 0
        print("classification error:", e)
    results.append({"text": t, "label": label, "anger": anger})
    tokens += used

import pandas as pd
df_reviews = pd.DataFrame(results)
print(df_reviews)
print("총 토큰:", tokens)
